# Fabric Anomaly Detection — Subspace Learning (PCA) on Deep Features

**Pipeline**

```
Normal Fabric Images -> Preprocessing -> Pretrained Feature Extractor (ResNet50)
   -> High-Dim Feature Vectors (2048-d) -> Subspace Learning (PCA/SVD)
   -> Learned Normal Subspace
Test Image -> Same Feature Extractor -> Feature Vector -> Project onto Subspace
   -> Reconstruction Error -> Anomaly Score -> Threshold -> Normal / Defective
```

**What this notebook does**

1. Loads your Kaggle fabric dataset (`train/good` for training, `test/{good,stain,hole,line}` for testing).
2. Extracts 2048-d global feature vectors with a frozen, ImageNet-pretrained **ResNet50**.
3. Learns a **normal subspace** with PCA on the *good* training features only (one-class / unsupervised setting — no defects are seen during training).
4. Scores every test image by its **PCA reconstruction error** (distance to the normal subspace).
5. Evaluates with **AUROC, ROC curve, confusion matrix, classification report, per-class score distributions**.
6. Also builds a **patch-level PCA subspace** (from an intermediate conv layer) so we can render **anomaly heatmaps** localizing the defect.

> ⚠️ **Before running:** edit the paths in the `CONFIG` cell to match your Kaggle dataset's actual folder structure. A helper cell is provided to print your dataset's folder tree so you can check/adjust the names.


In [ ]:

# ============================================================
# 1. Install / Import libraries
# ============================================================
import os, glob, random, math, json
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from tqdm.auto import tqdm
import joblib
import cv2

sns.set_style("whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:

# ============================================================
# 2. Helper: inspect your Kaggle dataset structure
#    Run this FIRST and use the printed tree to fix CONFIG below.
#
#    NOTE: Kaggle mounts dataset folders as SYMLINKS. os.walk() does NOT
#    follow symlinks by default, which silently reports "0 files" even
#    when the files exist (confirmed via `ls -la`). followlinks=True fixes it.
# ============================================================
KAGGLE_INPUT = "/kaggle/input"

for root_dataset in sorted(os.listdir(KAGGLE_INPUT)):
    print("Dataset folder:", root_dataset)
    base = os.path.join(KAGGLE_INPUT, root_dataset)
    depth_limit = 5
    for cur_root, dirs, files in os.walk(base, followlinks=True):
        depth = cur_root[len(base):].count(os.sep)
        if depth > depth_limit:
            dirs[:] = []
            continue
        indent = "  " * (depth + 1)
        n_files = len(files)
        sample = files[0] if files else ""
        print(f"{indent}{os.path.basename(cur_root)}/  ({n_files} files) {sample}")
    print()

# --- direct check on the exact dataset path you're using ---
MY_DATASET_PATH = "/kaggle/input/datasets/tasnimabrarsajin/dataset-for-fabric-anomaly-detection-for-training/EfficientNetB0"
print("=== Direct walk of your dataset path ===")
print(MY_DATASET_PATH, "exists:", os.path.isdir(MY_DATASET_PATH))
for cur_root, dirs, files in os.walk(MY_DATASET_PATH, followlinks=True):
    indent = "  " * (cur_root[len(MY_DATASET_PATH):].count(os.sep) + 1)
    n_files = len(files)
    sample = files[0] if files else ""
    print(f"{indent}{os.path.basename(cur_root)}/  ({n_files} files) {sample}")


## 3. Configuration

Edit `DATASET_ROOT` and the sub-folder names below so they match what you saw
printed above. The expected layout (MVTec-AD style) is:

```
DATASET_ROOT/
├── train/
│   └── good/            <- 1400 normal images
└── test/
    ├── good/
    ├── stain/
    ├── hole/             (folder might be named "holes")
    └── line/             (folder might be named "lines")
```


In [ ]:

# ============================================================
# 3. CONFIG — EDIT THESE PATHS TO MATCH YOUR DATASET
# ============================================================
DATASET_ROOT = "/kaggle/input/datasets/tasnimabrarsajin/dataset-for-fabric-anomaly-detection-for-training/EfficientNetB0"   # <-- EDIT ME if the tree printed above shows a different structure

TRAIN_GOOD_DIR = os.path.join(DATASET_ROOT, "train", "good")

TEST_DIRS = {
    "good":  os.path.join(DATASET_ROOT, "test", "good"),
    "stain": os.path.join(DATASET_ROOT, "test", "stain"),
    "hole":  os.path.join(DATASET_ROOT, "test", "hole"),   # rename to "holes" if needed
    "line":  os.path.join(DATASET_ROOT, "test", "line"),   # rename to "lines" if needed
}
# ^ If the tree from the diagnostic cell above shows different folder names
#   (e.g. "Train"/"Test", "Good"/"Stain", or an extra nesting level), edit
#   DATASET_ROOT and/or the individual TEST_DIRS values to match exactly
#   what was printed — folder names are case-sensitive on Kaggle's Linux FS.

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# Which defect classes count as the positive ("anomaly") class for metrics
DEFECT_CLASSES = ["stain", "hole", "line"]

VALID_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

# ============================================================
# 4. Build file lists
#
#    IMPORTANT: Kaggle mounts /kaggle/input as symlinks. os.walk() (and
#    os.listdir on a symlinked dir) will not traverse them unless
#    followlinks=True is passed, otherwise you silently get 0 images even
#    though the path is correct. This also recurses one or more levels deep
#    in case images sit in a nested subfolder rather than directly inside
#    "good"/"stain"/"hole"/"line".
# ============================================================
def list_images(folder):
    if not os.path.isdir(folder):
        print(f"[WARNING] Folder not found: {folder}")
        return []
    files = []
    for cur_root, dirs, fnames in os.walk(folder, followlinks=True):
        for f in fnames:
            if f.lower().endswith(VALID_EXTS):
                files.append(os.path.join(cur_root, f))
    files = sorted(files)
    if len(files) == 0:
        print(f"[WARNING] 0 images found under: {folder} "
              f"(folder exists, but is empty or contains no supported image files)")
    return files

train_files = list_images(TRAIN_GOOD_DIR)
print(f"Train (good) images: {len(train_files)}")

test_files, test_labels_str = [], []
for cls, folder in TEST_DIRS.items():
    files = list_images(folder)
    test_files.extend(files)
    test_labels_str.extend([cls] * len(files))
    print(f"Test [{cls:6s}] images: {len(files)}")

print(f"\nTotal test images: {len(test_files)}")

# binary ground truth: 0 = good/normal, 1 = defective
test_labels_bin = np.array([0 if c == "good" else 1 for c in test_labels_str])
test_labels_str = np.array(test_labels_str)

assert len(train_files) > 0, "No training images found — check DATASET_ROOT / TRAIN_GOOD_DIR."
assert len(test_files) > 0, "No test images found — check TEST_DIRS."


In [ ]:

# ============================================================
# 5. Dataset / Transforms
# ============================================================
weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()   # official resize + center-crop + ImageNet normalize

class FabricDataset(Dataset):
    def __init__(self, file_paths, transform):
        self.file_paths = file_paths
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), path

train_dataset = FabricDataset(train_files, preprocess)
test_dataset = FabricDataset(test_files, preprocess)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("Train batches:", len(train_loader), "| Test batches:", len(test_loader))


In [ ]:

# ============================================================
# 6. Frozen pretrained feature extractor (ResNet50)
#    - global feature vector (2048-d)  -> from avgpool (used for main AUROC pipeline)
#    - spatial feature map  (512xHxW)  -> from layer2   (used later for heatmaps)
# ============================================================
backbone = resnet50(weights=weights)
backbone.eval().to(device)
for p in backbone.parameters():
    p.requires_grad_(False)

_captured = {}

def _hook(name):
    def fn(module, inp, out):
        _captured[name] = out.detach()
    return fn

backbone.avgpool.register_forward_hook(_hook("global"))
backbone.layer2.register_forward_hook(_hook("spatial"))

@torch.no_grad()
def extract_global_features(loader):
    feats, paths = [], []
    for imgs, batch_paths in tqdm(loader, desc="Extracting global features"):
        imgs = imgs.to(device)
        _ = backbone(imgs)                       # hooks populate _captured
        g = _captured["global"].squeeze(-1).squeeze(-1)   # (B, 2048)
        feats.append(g.cpu().numpy())
        paths.extend(batch_paths)
    return np.concatenate(feats, axis=0), paths

print("Feature extractor ready. Global feature dim:", backbone.fc.in_features)


In [ ]:

# ============================================================
# 7. Extract 2048-d features for train (normal only) and test sets
# ============================================================
train_feats, train_paths = extract_global_features(train_loader)
test_feats, test_paths_ordered = extract_global_features(test_loader)

# test_loader is not shuffled, so test_paths_ordered aligns with test_files order,
# which aligns with test_labels_bin / test_labels_str built earlier.
assert test_paths_ordered == test_files

print("train_feats:", train_feats.shape)
print("test_feats :", test_feats.shape)


In [ ]:

# ============================================================
# 8. Subspace Learning Module — PCA on NORMAL features only
# ============================================================
# n_components as a float in (0,1) keeps enough components to explain that
# fraction of variance -> this defines the learned "normal subspace".
PCA_VARIANCE_KEEP = 0.95

subspace_pca = PCA(n_components=PCA_VARIANCE_KEEP, svd_solver="full", random_state=SEED)
subspace_pca.fit(train_feats)

print(f"Normal subspace dimensionality: {subspace_pca.n_components_} "
      f"(out of {train_feats.shape[1]} original dims)")
print(f"Explained variance captured: {subspace_pca.explained_variance_ratio_.sum():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(subspace_pca.explained_variance_ratio_))
plt.axhline(PCA_VARIANCE_KEEP, color="red", linestyle="--", label=f"{PCA_VARIANCE_KEEP:.0%} threshold")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA — Normal Subspace Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 9. Anomaly score = PCA reconstruction error (projection distance)
# ============================================================
def reconstruction_error(features, pca):
    projected = pca.transform(features)
    reconstructed = pca.inverse_transform(projected)
    errors = np.mean((features - reconstructed) ** 2, axis=1)  # per-sample MSE
    return errors

train_scores = reconstruction_error(train_feats, subspace_pca)
test_scores = reconstruction_error(test_feats, subspace_pca)

print("Train (normal) reconstruction error — "
      f"mean: {train_scores.mean():.6f}, std: {train_scores.std():.6f}, "
      f"max: {train_scores.max():.6f}")
print("Test reconstruction error — "
      f"mean: {test_scores.mean():.6f}, std: {test_scores.std():.6f}")


## 10. Evaluation

We report two complementary things:

1. **AUROC / ROC curve** — a threshold-free measure of how well the anomaly
   score separates *good* from *defective* fabric.
2. **A concrete decision threshold**, so we can compute a confusion matrix and
   classification report. We show two ways of picking it:
   - **Unsupervised threshold**: the *P*-th percentile of the *training* (all-normal)
     reconstruction errors — this is the realistic choice, since at deployment
     time you don't have labeled defects.
   - **Best (Youden's J) threshold from ROC on the test set** — the theoretical
     best possible operating point, shown for reference/comparison only (it "peeks"
     at test labels so it is optimistic).


In [ ]:

# ============================================================
# 11. AUROC + ROC curve
# ============================================================
auroc = roc_auc_score(test_labels_bin, test_scores)
fpr, tpr, roc_thresholds = roc_curve(test_labels_bin, test_scores)

print(f"AUROC = {auroc:.4f}")

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUROC = {auroc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Subspace (PCA) Anomaly Detector")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 12. Thresholding + Confusion Matrix + Classification Report
# ============================================================
# --- (a) Unsupervised threshold from TRAIN normal scores only ---
PERCENTILE = 99  # e.g. flag anything above the 99th percentile of normal errors
threshold_unsupervised = np.percentile(train_scores, PERCENTILE)

# --- (b) Best threshold via Youden's J statistic on the ROC curve (reference only) ---
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
threshold_best_roc = roc_thresholds[best_idx]

print(f"Unsupervised threshold (train {PERCENTILE}th percentile): {threshold_unsupervised:.6f}")
print(f"Best ROC (Youden's J) threshold [reference only]:        {threshold_best_roc:.6f}")

def evaluate_at_threshold(scores, y_true, threshold, title):
    y_pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report = classification_report(y_true, y_pred, target_names=["Normal", "Defective"],
                                    digits=4, zero_division=0)
    print(f"\n===== {title} (threshold = {threshold:.6f}) =====")
    print(report)

    fig, ax = plt.subplots(figsize=(5, 4.5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Defective"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return y_pred, cm, report

y_pred_unsup, cm_unsup, report_unsup = evaluate_at_threshold(
    test_scores, test_labels_bin, threshold_unsupervised,
    "Confusion Matrix — Unsupervised Threshold (train percentile)"
)

y_pred_best, cm_best, report_best = evaluate_at_threshold(
    test_scores, test_labels_bin, threshold_best_roc,
    "Confusion Matrix — Best ROC Threshold (reference only)"
)


In [ ]:

# ============================================================
# 13. Per-defect-type breakdown
# ============================================================
df_results = pd.DataFrame({
    "path": test_paths_ordered,
    "class": test_labels_str,
    "label_binary": test_labels_bin,
    "anomaly_score": test_scores,
    "pred_unsupervised": y_pred_unsup,
    "pred_best_roc": y_pred_best,
})

# Detection rate (recall) per class using the unsupervised threshold
per_class_recall = (
    df_results.groupby("class")
    .apply(lambda g: (g["pred_unsupervised"] == g["label_binary"]).mean()
           if g["class"].iloc[0] == "good"
           else (g["pred_unsupervised"] == 1).mean())
    .rename("accuracy_or_recall")
)
print(per_class_recall)

plt.figure(figsize=(7, 4))
sns.boxplot(data=df_results, x="class", y="anomaly_score",
            order=["good"] + DEFECT_CLASSES)
plt.axhline(threshold_unsupervised, color="red", linestyle="--", label="Unsupervised threshold")
plt.title("Anomaly Score Distribution by Class")
plt.ylabel("PCA Reconstruction Error (Anomaly Score)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
for cls in ["good"] + DEFECT_CLASSES:
    subset = df_results[df_results["class"] == cls]["anomaly_score"]
    if len(subset) > 0:
        sns.kdeplot(subset, label=cls, fill=True, alpha=0.25)
plt.axvline(threshold_unsupervised, color="red", linestyle="--", label="Unsupervised threshold")
plt.xlabel("Anomaly Score")
plt.title("Anomaly Score Density by Class")
plt.legend()
plt.tight_layout()
plt.show()

df_results.to_csv(os.path.join(OUTPUT_DIR, "test_scores.csv"), index=False)
print("Saved per-image scores to:", os.path.join(OUTPUT_DIR, "test_scores.csv"))


## 14. Anomaly Heatmaps (spatial / patch-level subspace)

The global 2048-d PCA above gives a single anomaly score per image (good for
AUROC/classification), but it can't show *where* the defect is. For that we
learn a **second, patch-level subspace** from the spatial feature map of an
earlier ResNet50 layer (`layer2`: 512 channels × 28×28 grid for a 224×224
input). Every spatial location (patch) is treated as its own 512-d feature
vector; we fit PCA on patches from normal training images, and at test time
compute the reconstruction error **per patch**, giving a 28×28 anomaly map
that we upsample and overlay on the original image.


In [ ]:

# ============================================================
# 15. Extract spatial (patch) features for the patch-level subspace
# ============================================================
PATCHES_PER_IMAGE_FOR_FITTING = 100  # subsample per training image to keep PCA fitting light

@torch.no_grad()
def extract_patch_features(loader, subsample_per_image=None):
    all_patches = []
    for imgs, _ in tqdm(loader, desc="Extracting spatial features"):
        imgs = imgs.to(device)
        _ = backbone(imgs)
        fmap = _captured["spatial"]                     # (B, C, H, W)
        B, C, H, W = fmap.shape
        fmap = fmap.permute(0, 2, 3, 1).reshape(B, H * W, C)  # (B, H*W, C)
        for b in range(B):
            patches = fmap[b].cpu().numpy()              # (H*W, C)
            if subsample_per_image is not None and subsample_per_image < patches.shape[0]:
                idx = np.random.choice(patches.shape[0], subsample_per_image, replace=False)
                patches = patches[idx]
            all_patches.append(patches)
    return np.concatenate(all_patches, axis=0), (H, W)

train_patch_feats, spatial_hw = extract_patch_features(
    train_loader, subsample_per_image=PATCHES_PER_IMAGE_FOR_FITTING
)
print("Patch feature matrix for fitting:", train_patch_feats.shape, "| spatial grid:", spatial_hw)


In [ ]:

# ============================================================
# 16. Patch-level PCA subspace
# ============================================================
patch_pca = PCA(n_components=PCA_VARIANCE_KEEP, svd_solver="randomized", random_state=SEED)
patch_pca.fit(train_patch_feats)
print(f"Patch subspace dimensionality: {patch_pca.n_components_} / {train_patch_feats.shape[1]}")
print(f"Explained variance captured: {patch_pca.explained_variance_ratio_.sum():.4f}")


In [ ]:

# ============================================================
# 17. Single-image anomaly heatmap function
# ============================================================
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def unnormalize_for_display(tensor_img):
    img = tensor_img.cpu().numpy().transpose(1, 2, 0)
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

@torch.no_grad()
def compute_anomaly_heatmap(image_path):
    img = Image.open(image_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)
    _ = backbone(x)
    fmap = _captured["spatial"]                       # (1, C, H, W)
    _, C, H, W = fmap.shape
    patches = fmap.permute(0, 2, 3, 1).reshape(H * W, C).cpu().numpy()

    projected = patch_pca.transform(patches)
    reconstructed = patch_pca.inverse_transform(projected)
    patch_errors = np.mean((patches - reconstructed) ** 2, axis=1)   # (H*W,)

    anomaly_map = patch_errors.reshape(H, W)
    anomaly_map_resized = cv2.resize(anomaly_map, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)

    display_img = unnormalize_for_display(x.squeeze(0))
    image_score = anomaly_map.max()   # patch-based image-level score (for illustration)

    return display_img, anomaly_map_resized, image_score

def show_heatmap(image_path, ax_img, ax_heat, title_prefix=""):
    display_img, amap, score = compute_anomaly_heatmap(image_path)
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)

    ax_img.imshow(display_img)
    ax_img.set_title(f"{title_prefix}\nscore={score:.4f}", fontsize=9)
    ax_img.axis("off")

    ax_heat.imshow(display_img)
    ax_heat.imshow(amap_norm, cmap="jet", alpha=0.45)
    ax_heat.set_title("Anomaly heatmap", fontsize=9)
    ax_heat.axis("off")


In [ ]:

# ============================================================
# 18. Visualize heatmaps: a few samples from each class
# ============================================================
N_SAMPLES_PER_CLASS = 3
classes_to_show = ["good"] + DEFECT_CLASSES

fig, axes = plt.subplots(len(classes_to_show), N_SAMPLES_PER_CLASS * 2,
                          figsize=(3 * N_SAMPLES_PER_CLASS * 2, 3.2 * len(classes_to_show)))
if len(classes_to_show) == 1:
    axes = axes[np.newaxis, :]

for row, cls in enumerate(classes_to_show):
    cls_paths = df_results[df_results["class"] == cls]["path"].tolist()
    sample_paths = random.sample(cls_paths, min(N_SAMPLES_PER_CLASS, len(cls_paths)))
    for col, path in enumerate(sample_paths):
        ax_img = axes[row, col * 2]
        ax_heat = axes[row, col * 2 + 1]
        show_heatmap(path, ax_img, ax_heat, title_prefix=f"[{cls}] {os.path.basename(path)}")
    # blank out any unused axes in this row
    for col in range(len(sample_paths), N_SAMPLES_PER_CLASS):
        axes[row, col * 2].axis("off")
        axes[row, col * 2 + 1].axis("off")

plt.tight_layout()
plt.show()


## 19. Summary & saving artifacts

In [ ]:

# ============================================================
# 20. Save everything needed to re-run inference later
# ============================================================
joblib.dump(subspace_pca, os.path.join(OUTPUT_DIR, "global_subspace_pca.joblib"))
joblib.dump(patch_pca, os.path.join(OUTPUT_DIR, "patch_subspace_pca.joblib"))

with open(os.path.join(OUTPUT_DIR, "metrics_summary.json"), "w") as f:
    json.dump({
        "auroc": float(auroc),
        "threshold_unsupervised_train_percentile": float(threshold_unsupervised),
        "percentile_used": PERCENTILE,
        "threshold_best_roc_reference_only": float(threshold_best_roc),
        "num_train_normal": int(len(train_files)),
        "num_test_total": int(len(test_files)),
        "test_class_counts": {c: int((test_labels_str == c).sum()) for c in np.unique(test_labels_str)},
        "global_subspace_dims": int(subspace_pca.n_components_),
        "patch_subspace_dims": int(patch_pca.n_components_),
    }, f, indent=2)

print("Saved:")
print(" -", os.path.join(OUTPUT_DIR, "global_subspace_pca.joblib"))
print(" -", os.path.join(OUTPUT_DIR, "patch_subspace_pca.joblib"))
print(" -", os.path.join(OUTPUT_DIR, "metrics_summary.json"))
print(" -", os.path.join(OUTPUT_DIR, "test_scores.csv"))
print(f"\nFINAL AUROC: {auroc:.4f}")
